# Motor Data Logging From Serial Bus

## Python Packages

In [8]:
import serial       # Used for serial communication over USB.
import time         # Used for timekeeping.
import os           # Used to check and access files.
import pandas as pd # Used to generate data structures.

## Configuration
- port: The port refers to the COM port or USB port the microcontroller is connected to.
- baud_rate: Set baud rate for communication, match this with the microcontroller code.
- csv_file: The name of the csv file for the generated data.
- CSV_folder_name: The name of the folder the csv shall be stored in.
- n_samples: The number of samples you wish to collect.

In [9]:
port = 'COM3'
baud_rate = 9600
csv_file = 'test.csv'
CSV_folder_name = "CSV_files"
n_samples = 100 

## Start recording

In [10]:
# Open the serial port
ser = serial.Serial(port, baud_rate, timeout=1)
print("Connection established.")


# Initialize empty lists to store the data
bus_voltage = []
shunt_voltage = []
current = []
angle = []
t = []


# Start time
start_time = time.time()
print("Starting recording...")


# Collect data
for i in range(n_samples):
    line = ser.readline().decode('utf-8').strip()
    if line:
        # Parse and store data
        data = line.split(',')
        if len(data) == 4:
            angle_raw = float(data[0])
            bus_voltage.append(float(data[1]))  # Convert to float
            shunt_voltage.append(float(data[2]))  # Convert to float
            current.append(float(data[3]))  # Convert to float
            angle.append(angle_raw)
            t.append(time.time() - start_time)

# Close the serial port
ser.close()


# Calculate total time spent logging
time_tot = time.time() - start_time


# Report
print("Recording Complete.")
print("Time spent logging: {:.2f} seconds".format(time_tot))
print("Storing data...")

Connection established.
Starting recording...
Recording Complete.
Time spent logging: 3.36 seconds
Storing data...


## Save data and report

In [11]:
# Create a DataFrame from the collected data
df = pd.DataFrame({
    'Angle': angle,
    'V_bus': bus_voltage,
    'V_shunt': shunt_voltage,
    'current': current,
    'time': t
})


# Check if the file exists, and append or create accordingly
if os.path.isfile(csv_file):  # File exists
    # Read the CSV file into a DataFrame
    df_old = pd.read_csv(csv_file)

    # Concatenate the old DataFrame with the new one
    df = pd.concat([df_old, df], axis=1)


# Save data
file_path = os.path.join(CSV_folder_name, csv_file) # Generate file path
df.to_csv(file_path, index=False) # Save df


# Report
print("-------------------------------------------------------------------")
print("Data saved to {}: {} samples recorded in {:.2f} seconds".format(csv_file, n_samples, time_tot))
print("-------------------------------------------------------------------")

-------------------------------------------------------------------
Data saved to test.csv: 100 samples recorded in 3.36 seconds
-------------------------------------------------------------------
